In [92]:
import math, itertools, random
import numpy as np
import pandas as pd

# Try imports for solvers
has_ortools = False
has_pulp = False
try:
    from ortools.linear_solver import pywraplp
    has_ortools = True
except Exception as e:
    try:
        import pulp
        has_pulp = True
    except Exception as e2:
        pass

import torch
import torch.nn as nn
import torch.nn.functional as F


from importnb import Notebook
with Notebook():
    from LabTrajectory import simulate_viewport_with_tiles
    from LabCacheEngine import CacheEngineEnv, LruPolicy
    from LabLatencyModel import LatencyModel

In [93]:
class SimpleDQN(nn.Module):
    def __init__(self, input_dim, hidden=[64,32]):
        super().__init__()
        layers = []
        cur = input_dim
        for h in hidden:
            layers.append(nn.Linear(cur,h))
            layers.append(nn.ReLU())
            cur = h
        layers.append(nn.Linear(cur,1))  # scalar Q for a (s,a) pair
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)  # (...,) shape

In [94]:
# ----------------------
# Fake environment & features for demonstration
# ----------------------
# random.seed(0)
# np.random.seed(0)
# torch.manual_seed(0)

# Suppose we have 8 candidate tiles/items to possibly cache
n_items = 8

# For each item, define:
sizes = np.random.randint(1,6, size=n_items)           # storage size units
latency_if_not_cached = np.random.uniform(50,500,size=n_items)  # ms cost if not cached
benefit = latency_if_not_cached  # simplifying: benefit of caching equals saved latency
capacity = 15   # total storage units available at the cache

# Feature vector for each (s,a) pair: we'll use [size, recent_requests, popularity_est, benefit]
recent_requests = np.random.randint(0,20,size=n_items)
popularity = np.random.rand(n_items)

features = np.stack([sizes, recent_requests, popularity*100, benefit], axis=1).astype(np.float32)
# Normalize features (simple)
features = (features - features.mean(axis=0)) / (features.std(axis=0) + 1e-6)

In [95]:
# Create a DQN and compute Q-values for each candidate
dqn = SimpleDQN(input_dim=features.shape[1])
# random init is fine for demo; in practice load trained weights
with torch.no_grad():
    x = torch.tensor(features)
    q_values = dqn(x).numpy()

# Choose threshold phi to create DQN mask (tunable)
phi = np.percentile(q_values, 60)   # keep top 40% suggested items as mask
mask = (q_values >= phi).astype(int)

working_set = np.arange(n_items)



In [96]:
force_one = np.zeros(n_items, dtype=int)
force_zero = np.zeros(n_items, dtype=int)
for i in working_set:
    if mask[i]==1:
        force_one[i]=1
    else:
        force_zero[i]=1

In [97]:
def solve_pruned_ilp_ortools(sizes, benefit, capacity, force_one, force_zero):
    solver = pywraplp.Solver.CreateSolver('SCIP') or pywraplp.Solver.CreateSolver('CBC_MIXED_INTEGER_PROGRAMMING')
    if solver is None:
        raise RuntimeError("OR-Tools solver not available")
    n = len(sizes)
    L = [solver.IntVar(0,1,f"L_{i}") for i in range(n)]
    # capacity
    solver.Add(sum(sizes[i]*L[i] for i in range(n)) <= capacity)
    # force constraints
    for i in range(n):
        if force_one[i]:
            solver.Add(L[i] == 1)
        if force_zero[i]:
            solver.Add(L[i] == 0)
    # maximize benefit (equiv to minimize latency)
    objective = solver.Objective()
    for i in range(n):
        objective.SetCoefficient(L[i], benefit[i])
    objective.SetMaximization()
    status = solver.Solve()
    if status == pywraplp.Solver.OPTIMAL or status == pywraplp.Solver.FEASIBLE:
        sol = np.array([int(L[i].solution_value()) for i in range(n)])
        return sol, solver.Objective().Value()
    else:
        raise RuntimeError("No solution found by OR-Tools")

In [98]:
def solve_pruned_ilp_pulp(sizes, benefit, capacity, force_one, force_zero):
    import pulp
    prob = pulp.LpProblem("pruned_cache", pulp.LpMaximize)
    n = len(sizes)
    L = [pulp.LpVariable(f"L_{i}", cat='Binary') for i in range(n)]
    prob += pulp.lpSum([benefit[i]*L[i] for i in range(n)])
    prob += pulp.lpSum([sizes[i]*L[i] for i in range(n)]) <= capacity
    for i in range(n):
        if force_one[i]:
            prob += L[i] == 1
        if force_zero[i]:
            prob += L[i] == 0
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    sol = np.array([int(pulp.value(L[i])) for i in range(n)])
    obj = pulp.value(prob.objective)
    return sol, obj

In [99]:
def solve_pruned_ilp_bruteforce(sizes, benefit, capacity, force_one, force_zero):
    n = len(sizes)
    best_obj = -1e9
    best_sol = None
    # brute force over subsets (only ok for n<=20)
    for bits in range(1<<n):
        sel = np.array([1 if (bits>>i)&1 else 0 for i in range(n)], dtype=int)

        # check forced constraints
        if np.any((force_one==1) & (sel==0)): continue
        if np.any((force_zero==1) & (sel==1)): continue
        if sel.dot(sizes) > capacity: continue

        obj = sel.dot(benefit)
        if obj > best_obj:
            best_obj = obj
            best_sol = sel.copy()
    if best_sol is None:
        raise RuntimeError("No feasible solution found by brute force")
    return best_sol, best_obj

force_one = np.zeros(n_items, dtype=int)
force_zero = np.zeros(n_items, dtype=int)
for i in working_set:
    if mask[i]==1:
        force_one[i]=1
    else:
        force_zero[i]=1

In [100]:
sol, obj = solve_pruned_ilp_pulp(sizes, benefit, capacity, force_one, force_zero)
solver_used = "OR-Tools"

# sol, obj = solve_pruned_ilp_ortools(sizes, benefit, capacity, force_one, force_zero)
# solver_used = "BruteForce (fallback)"

# sol, obj = solve_pruned_ilp_bruteforce(sizes, benefit, capacity, force_one, force_zero)
# solver_used = "PuLP"

In [101]:
# Compute final latency given selection: sum latency_if_not_cached * (1 - L)
final_latency = np.sum(latency_if_not_cached * (1 - sol))

# Present results
df = pd.DataFrame({
    'item': np.arange(n_items),
    'size': sizes,
    'latency_if_not_cached_ms': np.round(latency_if_not_cached,2),
    'benefit': np.round(benefit,2),
    'recent_requests': recent_requests,
    'popularity': np.round(popularity,3),
    'q_value': np.round(q_values,4),
    'mask_phi': mask,
    'forced_one': force_one,
    'forced_zero': force_zero,
    'chosen_L': sol
})

# Try to use optional helper if available; otherwise display and save CSV
try:
    import caas_jupyter_tools as cjt
    cjt.display_dataframe_to_user("pruned_ilp_results", df)
    print("Displayed via caas_jupyter_tools.")
except Exception:
    try:
        from IPython.display import display
        display(df)
        print("Displayed via IPython.display. Also saving CSV...")
    except Exception:
        print("Falling back to CSV only...")
    df.to_csv("pruned_ilp_results.csv", index=False)
    print("Saved results to pruned_ilp_results.csv")

print(f"Solver used: {solver_used}")
print(f"Capacity: {capacity}, total sizes chosen: {int(np.dot(sol,sizes))}")
print(f"Objective (sum benefit of cached items): {obj:.2f}")
print(f"Final total latency (ms) after caching selection: {final_latency:.2f}")


,item,size,latency_if_not_cached_ms,benefit,recent_requests,popularity,q_value,mask_phi,forced_one,forced_zero,chosen_L
0,0,3,199.53,199.53,15,0.385,-0.1274,1,1,0,1
1,1,5,495.82,495.82,2,0.387,-0.1920,0,0,1,0
2,2,4,211.83,211.83,3,0.103,-0.1838,0,0,1,0
3,3,4,476.59,476.59,17,0.389,-0.1371,1,1,0,1
4,4,1,90.76,90.76,14,0.969,-0.1950,0,0,1,0
5,5,1,136.55,136.55,7,0.374,-0.1914,0,0,1,0
6,6,5,397.95,397.95,17,0.263,-0.1442,1,1,0,1
7,7,4,426.59,426.59,6,0.448,-0.2041,0,0,1,0


Displayed via IPython.display. Also saving CSV...
Saved results to pruned_ilp_results.csv
Solver used: OR-Tools
Capacity: 15, total sizes chosen: 12
Objective (sum benefit of cached items): 1074.07
Final total latency (ms) after caching selection: 1361.55


In [102]:
from ortools.sat.python import cp_model

In [103]:
def build_pruned_ilp(
        working_tiles,      # Tiles in Sw
        replacement_tiles,  # Tiles in Sr
        dqn_mask,           # result from mask: 1 or 0 per action
        latency_costs,      # dict {tile: cost}
        capacity_limit
):
    # Create the CP-SAT model
    model = cp_model.CpModel()

    # ILP Variables
    # L_tile = 1 means tile is placed in cache
    L = {t: model.NewBoolVar(f"L_{t}") for t in working_tiles + replacement_tiles}

    # A_i = action selector (which action we pick)
    A = [model.NewBoolVar(f"a_{i}") for i in range(len(dqn_mask))]

    # --- (38) Cutting plane: A_i <= DQN_mask_i ---
    for i, m in enumerate(dqn_mask):
        if m == 0:
            model.Add(A[i] == 0)
        # if m == 1 -> leave A_i free until global constraints decide

    # --- (39) Exactly one action chosen ---
    model.Add(sum(A) == 1)

    # --- (36) Keep tiles in working set ---
    for t in working_tiles:
        model.Add(L[t] == 1)

    # --- (37) Remove tiles in replacement set ---
    for t in replacement_tiles:
        model.Add(L[t] == 0)

    # Capacity constraint (generic placeholder)
    # adjust as needed (e.g., sum(sizes[t] * L[t] for t in L) <= capacity_limit)
    model.Add(sum(L.values()) <= capacity_limit)

    # Objective: Minimize total latency cost
    model.Minimize(
        sum(latency_costs[t] * L[t] for t in L)
    )

    return model, L, A

In [104]:
def solve_ilp(model, L, A):
    # Create the solver and solve
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 0.3  # optional time limit
    solver.parameters.num_search_workers = 8  # parallelism
    
    result = solver.Solve(model)

    if result not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        raise RuntimeError("No solution found by CP-SAT solver")
    
    chosen_action = [i for i, a in enumerate(A) if solver.Value(a) == 1]
    placement = {t: solver.Value(var) for t, var in L.items()}

    return chosen_action[0], placement

In [105]:
working_tiles = [0,1,2,3]
replacement_tiles = [4,5,6,7]
latency_costs = {i: 5 for i in range(8)}
capacity_limit = 4

dqn_mask = [1,0,1,1,0,0,1,0]  # example mask from DQN

model, L, A = build_pruned_ilp(
    working_tiles, 
    replacement_tiles, 
    dqn_mask,
    latency_costs, 
    capacity_limit
)

chosen_action, placement = solve_ilp(model, L, A)

print(f"Chosen action: {chosen_action}")
print(f"Placement: {placement}")

Chosen action: 0
Placement: {0: 1, 1: 1, 2: 1, 3: 1, 4: 0, 5: 0, 6: 0, 7: 0}


In [106]:
# ==========================================================
# 0) PARAMETERS (change to match your real system)
# ==========================================================
TILES_X = 4
TILES_Y = 4
N_TILES = TILES_X * TILES_Y   # 24 tiles in 360° view
CACHE_SIZE = 10               # DU can store up to 10 tiles
ACTION_DIM = N_TILES          # each action = "evict tile i"
STATE_DIM = N_TILES * 2       # viewport probs + occupancy

In [107]:
# ==========================================================
# 1) VIEWPORT PREDICTOR (your model would go here)
# ==========================================================
def viewport_predictor(user_pose=None):
    """
    Dummy predictor: returns probabilities for each of 24 tiles.
    Replace this with your actual viewport predictor.
    """
    probs = np.random.rand(N_TILES)
    probs /= probs.sum()    # normalize
    return probs

In [108]:
# ==========================================================
# 1.1) VIEWPORT PROBABILITIES BASED ON SYNTHETIC DATA
# ==========================================================
def viewport_probs_synthetic(
    n_users: int = 10,
    n_gops: int = 60,
    n: int = 4,
    step_size: float = 5.0,
    fov__yaw: float = 90,
    fov_pitch: float = 50
):
    user_required_tiles = []
    for f in range(n_users):
        _, _, tiles_per_frame, _ = simulate_viewport_with_tiles(
            num_steps=n_gops,
            n=n,
            fov_yaw=fov__yaw,
            fov_pitch=fov_pitch,
            step_size=step_size,
            damping=1.0,
            start_yaw=180,
            start_pitch=0
        )
        user_required_tiles += tiles_per_frame

    tiles_grid = np.zeros((n,n))
    for tiles in user_required_tiles:
        for (x,y) in tiles:
            if 0 <= x < n and 0 <= y < n:
                tiles_grid[x,y] += 1

    counts = tiles_grid.flatten()
    counts = counts + 1e-6
    
    # Normalize to sum to exactly 1.0
    probs = counts / counts.sum()

    return probs


In [109]:
# ==========================================================
# 2) BUILD STATE VECTOR
# ==========================================================
def build_state(viewport_probs, cache_bitmap):
    """
    State = concat( viewport_probabilities, cache_occupancy )
    """
    return np.concatenate([viewport_probs, cache_bitmap]).astype(np.float32)

In [110]:
# ==========================================================
# 3) DQN Network
# ==========================================================
class DQN(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(s_dim, 256), nn.ReLU(),
            nn.Linear(256, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, a_dim)
        )

    def forward(self, s):
        return self.net(s)

In [111]:
# ==========================================================
# 4) DQN Inference + Mask (thresholded Q-values)
# ==========================================================
def dqn_mask(Q_values, mode="top_k_4"):
    """
    Q_values: tensor with shape [A]
    mode: threshold rule
    Returns: binary mask
    """
    if mode == "top_k_4":
        k = min(4, len(Q_values))
        thr = torch.topk(Q_values, k).values[-1]
    else:
        raise ValueError("Unsupported threshold mode.")

    mask = (Q_values >= thr).int()
    return mask, float(thr)

In [112]:
# ==========================================================
# 5) ILP Cache Replacement Solver (pruned by DQN mask)
# ==========================================================

def build_and_solve_ilp(cache_bitmap, new_tile, dqn_mask):
    """
    cache_bitmap: 24-length 0/1 vector
    new_tile: integer index of tile to insert
    dqn_mask: 24-length mask from DQN
    """

    model = cp_model.CpModel()

    # ILP Variables
    # L[i] = final occupancy after replacement
    L = [model.NewBoolVar(f"L_{i}") for i in range(N_TILES)]

    # A[i] = action selector (which tile to evict)
    A = [model.NewBoolVar(f"a_{i}") for i in range(ACTION_DIM)]


    # --- (38) Cutting plane: A_i <= DQN_mask_i ---
    for i in range(ACTION_DIM):
        if dqn_mask[i] == 0:
            model.Add(A[i] == 0)

    # --- (39) Exactly one action chosen ---
    model.Add(sum(A) == 1)

    for i in range(N_TILES):
        if cache_bitmap[i] == 1:
            pass
            # Tile is currently cached
        #     model.Add(L[i] == 1 - A[i])  # Evict if action chosen
        # else:
        #     # Tile is not cached
        #     if i == new_tile:
        #         model.Add(L[i] == 1)  # New tile must be cached
        #     else:
        #         model.Add(L[i] == 0)  # Remain uncached

    # Inserto eviction: if A[i] == 1, then L[i] == 0
    for i in range(N_TILES):
        model.Add(L[i] == 0).OnlyEnforceIf(A[i])

    model.Add(L[new_tile] == 1)  # New tile must be cached

    model.Add(sum(L) <= CACHE_SIZE)

    # Objective: keep tiles with highest viewport probability
    # (In real system: use latency or utility from paper)
    vp = viewport_predictor()
    model.Minimize(-sum(int(vp[i] * 1000) * L[i] for i in range(N_TILES)))

    # Solve -----------------------------------------------------
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = 0.1
    solver.parameters.num_search_workers = 8

    result = solver.Solve(model)

    if result not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        return None, None

    chosen_action = None
    for i in range(N_TILES):
        if solver.Value(A[i]) == 1:
            chosen_action = i

    final_cache = np.array([solver.Value(L[i]) for i in range(N_TILES)])
    return chosen_action, final_cache

In [113]:
# ==========================================================
# 6) END-TO-END EXECUTION
# ==========================================================
def run_demo():
    print("\n=== END-TO-END 360° CACHE REPLACEMENT DEMO ===")

    # Step 1: Get viewport prediction
    viewport_probs = viewport_predictor()

    # Step 2: Build cache bitmap (fake initial cache)
    cache_bitmap = np.zeros(N_TILES)
    cache_bitmap[:CACHE_SIZE] = 1  # first 10 tiles cached

    # Step 3: Build state vector
    s_t = build_state(viewport_probs, cache_bitmap)

    # Step 4: Create and run DQN
    model = DQN(STATE_DIM, ACTION_DIM)
    with torch.no_grad():
        q_vals = model(torch.tensor(s_t).unsqueeze(0)).squeeze(0)

    mask, thr = dqn_mask(q_vals)

    print("\nDQN Q-values:")
    print(q_vals)
    print("\nThreshold φ:", thr)
    print("Mask (1=allowed action):")
    print(mask.numpy())

    # Step 5: Assume we need to insert a tile requested by viewport predictor
    new_tile = int(np.argmax(viewport_probs))
    print("\nTile requested (highest viewport probability):", new_tile)

    # Step 6: ILP replacement
    chosen_evict, final_cache = build_and_solve_ilp(cache_bitmap, new_tile, mask)

    print("\n### ILP RESULT ###")
    print("Tile evicted:", chosen_evict)
    print("Cache before:", cache_bitmap.astype(int))
    print("Cache after:", final_cache.astype(int))

    return chosen_evict, final_cache

In [114]:
# Run the pipeline
if __name__ == "__main__":
    run_demo()



=== END-TO-END 360° CACHE REPLACEMENT DEMO ===

DQN Q-values:
tensor([-0.0581,  0.0539, -0.0868,  0.1307, -0.0183,  0.0969, -0.0501,  0.1081,
        -0.0737, -0.1336,  0.0562,  0.2416, -0.0197,  0.0432,  0.0798,  0.0120])

Threshold φ: 0.09689037501811981
Mask (1=allowed action):
[0 0 0 1 0 1 0 1 0 0 0 1 0 0 0 0]

Tile requested (highest viewport probability): 8

### ILP RESULT ###
Tile evicted: 7
Cache before: [1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0]
Cache after: [1 1 1 1 0 1 1 0 1 0 1 0 1 1 0 0]


In [115]:
vp = viewport_probs_synthetic(
    n_users=1_000,
    n_gops=1800,
    n=4,
    step_size=5.0,
    fov__yaw=90,
    fov_pitch=50
)
vp
print("Viewport probabilities shape:", vp.shape)
print("Sum should be 1:", vp.sum())
print("Top 5 tiles likely to be watched:")
top5 = np.argsort(vp)[-4:]

for t in top5:
    print(f"Tile {t}: prob={vp[t]:.4f}")

Viewport probabilities shape: (16,)
Sum should be 1: 1.0
Top 5 tiles likely to be watched:
Tile 4: prob=0.0787
Tile 11: prob=0.0796
Tile 8: prob=0.0804
Tile 7: prob=0.0808


In [116]:
# DEBUG STEP 2 — cache bitmap
cache_bitmap = np.zeros(N_TILES)
cache_bitmap[:CACHE_SIZE] = 1

print("Cache bitmap:", cache_bitmap.astype(int))
print("Cache contains tiles:", np.where(cache_bitmap == 1)[0])

cache_env = CacheEngineEnv(n_tiles=16, n_layers=2, n_videos=100, cache_capacity=50e6)
state, info = cache_env.reset()

Cache bitmap: [1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0]
Cache contains tiles: [0 1 2 3 4 5 6 7 8 9]


In [117]:
s_t = build_state(vp, cache_bitmap)
s_t

array([0.06390885, 0.03889784, 0.03985989, 0.06749134, 0.07865111,
       0.06380014, 0.06566969, 0.08084469, 0.08044586, 0.06471219,
       0.06512902, 0.07955502, 0.06570359, 0.03980989, 0.03931921,
       0.06620166, 1.        , 1.        , 1.        , 1.        ,
       1.        , 1.        , 1.        , 1.        , 1.        ,
       1.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        ], dtype=float32)

In [118]:
import math
import random
import numpy as np
import gymnasium as gym

from time import sleep
from collections import deque
from itertools import count

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from importnb import Notebook
with Notebook():
    from LabLatencyModel import LatencyModel
    from LabCacheEngine import CacheEngineEnv
    from LabUserTileRequest import UserTileRequestEvents
    from LabEnvWrapper import EnvWrapper

In [119]:
if __name__ == "__main__":
    n_episodes = 200
    n_users = 100
    arrival_rate = 5.0  # users per second
    max_users_capacity = n_users
    alpha = 1.0
    n_videos = 100
    n_gops = 60
    n_layers = 2
    n = 4
    n_tiles = n * n
    # max_capacity = 50e6  # 50 MB
    max_capacity = (2 + 15) * n_videos * n_gops * 0.10 * 1e6  # 14 MB per video, 100 videos, 180 GOPs, 10% cache ratio, in bytes 

    # Hyperparameters for RL
    epsilon_start = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.995
    gamma = 0.99
    learning_rate = 1e-3
    batch_size = 64
    capacity = 10000
    seq_len = 2  # LSTM sequence length (history window)

    # CPT parameters
    theta = 0.5
    lam = 3.7183


    # Initialize User Environment
    users_env = UserTileRequestEvents(
        n_users=n_users,
        n_videos=n_videos,
        n_gops=n_gops,
        n_layers=n_layers,
        n_tiles=n_tiles,
        n=n,
        users_viewport_tiles=None,
        requested_videos=None,
        users_arrivals=None,
        arrival_rate=arrival_rate
    )

    cache_env = CacheEngineEnv(
        n_tiles=n*n,
        n_videos=n_videos,
        cache_capacity=max_capacity,
    )

    # Create latency model (replace numbers with your real config)
    P = 1; max_U = n_users
    lat_model = LatencyModel(
        P=P, 
        max_U=max_U,
        R_M_D=80e6,   # 640 Mbps -> 80e6 B/s 
        R_C_M=1.25e9, # 10 Gbps -> 1.25e9 B/s
        mu=2e7, 
        eta=2e5,
        B_pu_matrix=np.full((P, max_U), 40e6, dtype=float),       # 320 Mbps -> 40e6 B/s
        gamma_pu_matrix=np.full((P, max_U), 5.0, dtype=float),    # SNRs
        rhoT_p=[0.2], 
        lambda_p=[0.05],
        du_fixed_delay=0.001,    # 1 ms
        mec_fixed_delay=0.005,   # 5 ms
        cloud_fixed_delay=0.05   # 50 ms
    )

    env = EnvWrapper(
        n=n,
        n_layers=n_layers,
        users_env=users_env, 
        cache_env=cache_env, 
        latency_model=lat_model,
        theta=theta,
        lam=lam
    )

    print(
        f"Experiment ===================\n"
        f"Total users: {n_users}\n"
        f"Cache capacity: {max_capacity}MB\n"
        f"Video matrix: {n_videos}x{n_layers}x{n*n}\n"
        f"==============================\n"
    )

    for episode in range(n_episodes):
        state, info = env.reset()
        total_reward = 0.0
        
        print(f"\n--- Episode {episode+1} ---")
        print(info["cache"].shape)
        with np.printoptions(threshold=np.inf):
            print(info["cache"])

        for t in count():
            action = env.sample_action()

            next_state, reward, done, info = env.step(action)
            total_reward += reward

            if done:
                break

        print(f"Episode {episode+1}/{n_episodes}, Total Reward: {total_reward:.2f}")

Experiment ===================
Total users: 100
Cache capacity: 10200000000.0MB
Video matrix: 100x2x16


--- Episode 1 ---
(100, 2, 16)
[[[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]]

 [[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]
  [1 1 1 1 1 1

KeyboardInterrupt: 